# Exoplanet Pipeline â€” Model Lab

Train/test the **trained** models from the handoff (`Claude method -1.md` Â§3) *before* building the app.
The detection layer (TLS/BLS/Lomb-Scargle/ACF) is classical and untrained â€” not here.
Parameter estimation (batman/MCMC/GP) is fitting, not classification â€” not here.

This notebook covers the three things that actually learn:

| Model | Task | Sampling concern |
|---|---|---|
| **A. Per-dip classifier** (CORE) | multiclass `transit / EB / blend` | leakage: dips from one star must not split across train/test; class imbalance |
| **B. Spot/rotation detector** | binary `rotation yes/no` | different feature representation (LS/ACF); own split |
| **C. OOD / novelty gate** | inlier vs `unknown` | trained on **inliers only**; evaluated on held-out novelties |

Each model gets its own correct sampling and its own evaluation block. A final cell collates metrics.

**Data**: runs now on a synthetic feature generator so the whole pipeline is exercisable today. Replace `load_features()` with your real per-dip feature CSV when it exists â€” schema is documented in that cell.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    precision_recall_curve, brier_score_loss, roc_curve,
)

RNG = 42
np.random.seed(RNG)

# Optional libs â€” use if installed, else fall back to sklearn. Don't make them hard deps.
try:
    from lightgbm import LGBMClassifier
    HAVE_LGBM = True
except Exception:
    HAVE_LGBM = False
try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False
try:
    from imblearn.over_sampling import SMOTE
    HAVE_SMOTE = True
except Exception:
    HAVE_SMOTE = False

print(f'LightGBM={HAVE_LGBM}  XGBoost={HAVE_XGB}  SMOTE={HAVE_SMOTE}')

## 1. Data loading

Real input = one row per **detected dip** with the physical vetting features (Â§9). Required columns:

- `star_id` â€” grouping key (prevents leakage). `sector` â€” for the sector-split generalization test.
- per-dip features: `depth_ppm`, `duration_hr`, `odd_even_diff`, `secondary_depth`, `trap_shape` (0=U/flat planet â†’ 1=V/grazing EB), `centroid_shift`, `rho_consistency`, `snr`, `sde`, `contratio`.
- spot-channel features (per star, broadcast to its dips): `ls_power`, `acf_peak`, `rot_amplitude`.
- labels: `dip_class` in {transit, EB, blend, unknown}; `rotation` in {0,1}.

`unknown` rows exist only to test the OOD gate â€” they are never used to train the supervised classifier.

Drop in real data by replacing the body of `load_features()` with `pd.read_csv(...)`. Synthetic fallback is class-conditional so models have real signal to learn.

In [ ]:
DIP_FEATURES = ['depth_ppm','duration_hr','odd_even_diff','secondary_depth',
                'trap_shape','centroid_shift','rho_consistency','snr','sde','contratio']
SPOT_FEATURES = ['ls_power','acf_peak','rot_amplitude']

def _synth(n_stars=400, sectors=(1,2,3,4), seed=RNG):
    """Class-conditional synthetic per-dip features. ponytail: stand-in until real feature CSV exists."""
    rng = np.random.default_rng(seed)
    rows = []
    # planets are rare (handoff Â§10): heavy imbalance on purpose so the sampling code is actually tested
    class_p = {'transit': 0.12, 'EB': 0.30, 'blend': 0.18, 'unknown': 0.40}
    for s in range(n_stars):
        sector = rng.choice(sectors)
        rotating = rng.random() < 0.45
        ls = rng.gamma(4, 1.5) + (6 if rotating else 0)
        acf = rng.beta(5, 2) if rotating else rng.beta(1.5, 5)
        amp = (rng.gamma(2, 0.4) if rotating else rng.gamma(1.2, 0.05))
        for _ in range(rng.integers(1, 4)):  # 1-3 dips per star
            cls = rng.choice(list(class_p), p=list(class_p.values()))
            if cls == 'transit':       # shallow, flat-bottomed, no secondary, density-consistent
                f = dict(depth_ppm=rng.lognormal(7.0,0.6), duration_hr=rng.gamma(4,0.7),
                         odd_even_diff=abs(rng.normal(0,0.02)), secondary_depth=abs(rng.normal(0,30)),
                         trap_shape=rng.beta(2,6), centroid_shift=abs(rng.normal(0,0.05)),
                         rho_consistency=rng.normal(0,0.4), snr=rng.gamma(5,2), sde=rng.gamma(6,1.5),
                         contratio=rng.gamma(1.5,0.05))
            elif cls == 'EB':          # deep, secondary present, odd-even diff, V-shape
                f = dict(depth_ppm=rng.lognormal(9.5,0.7), duration_hr=rng.gamma(5,1.0),
                         odd_even_diff=abs(rng.normal(0,0.25))+0.05, secondary_depth=abs(rng.normal(800,400)),
                         trap_shape=rng.beta(6,2), centroid_shift=abs(rng.normal(0,0.1)),
                         rho_consistency=rng.normal(2.0,1.0), snr=rng.gamma(8,3), sde=rng.gamma(9,2),
                         contratio=rng.gamma(1.5,0.05))
            elif cls == 'blend':       # diluted depth, centroid shift, high contamination
                f = dict(depth_ppm=rng.lognormal(8.0,0.8), duration_hr=rng.gamma(4,0.9),
                         odd_even_diff=abs(rng.normal(0,0.1)), secondary_depth=abs(rng.normal(200,200)),
                         trap_shape=rng.beta(3,3), centroid_shift=abs(rng.normal(0.6,0.2))+0.2,
                         rho_consistency=rng.normal(1.0,1.2), snr=rng.gamma(4,2), sde=rng.gamma(5,1.5),
                         contratio=rng.gamma(4,0.15)+0.3)
            else:                      # unknown: drawn off-manifold (broad/odd) -> should trip OOD gate
                f = dict(depth_ppm=rng.lognormal(8.5,1.6), duration_hr=rng.gamma(2,3.0),
                         odd_even_diff=abs(rng.normal(0,0.5)), secondary_depth=abs(rng.normal(0,1500)),
                         trap_shape=rng.random(), centroid_shift=abs(rng.normal(0,1.0)),
                         rho_consistency=rng.normal(0,4.0), snr=rng.gamma(3,4), sde=rng.gamma(3,3),
                         contratio=rng.gamma(2,0.4))
            f.update(star_id=f'TIC{s:05d}', sector=int(sector), dip_class=cls,
                     rotation=int(rotating), ls_power=ls, acf_peak=acf, rot_amplitude=amp)
            rows.append(f)
    return pd.DataFrame(rows)

def load_features():
    # return pd.read_csv('features.csv')   # <-- swap in real data here
    return _synth()

df = load_features()
print(df.shape)
print(df['dip_class'].value_counts(normalize=True).round(3).to_dict())
df.head()

In [ ]:
# Sanity checks at the trust boundary â€” fail loud, not silent.
required = set(DIP_FEATURES + SPOT_FEATURES + ['star_id','sector','dip_class','rotation'])
missing = required - set(df.columns)
assert not missing, f'missing columns: {missing}'
assert df[DIP_FEATURES].isna().sum().sum() == 0, 'NaNs in dip features â€” impute or drop before training'
assert set(df['dip_class']) <= {'transit','EB','blend','unknown'}, 'unexpected class label'
print('schema OK')

## 2. Sampling â€” leakage-safe splits (the part that's easy to get wrong)

Two rules from handoff Â§10:
1. **Split by star** â€” multiple dips share a `star_id`; if some land in train and some in test, the model memorizes the star, not the physics. Use group-aware splits.
2. **Split by sector** â€” hold out whole sectors to simulate *future data* / different instrument systematics.

Imbalance handling (SMOTE/over-sampling) is applied **inside the training fold only**, after the split â€” never before, or the test set leaks into resampling.

In [ ]:
def group_holdout(frame, group_col='star_id', test_size=0.25, seed=RNG):
    """Single train/test split, no group spans both sides."""
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    tr, te = next(gss.split(frame, groups=frame[group_col]))
    return frame.iloc[tr].copy(), frame.iloc[te].copy()

def sector_holdout(frame, test_sectors):
    """Hold out entire sectors â€” the 'works on future data' test."""
    te = frame[frame['sector'].isin(test_sectors)].copy()
    tr = frame[~frame['sector'].isin(test_sectors)].copy()
    return tr, te

def resample_train(X, y, seed=RNG):
    """Balance the TRAINING fold only. SMOTE if available, else return as-is (models use class_weight)."""
    if HAVE_SMOTE and pd.Series(y).value_counts().min() >= 6:
        return SMOTE(random_state=seed, k_neighbors=5).fit_resample(X, y)
    return X, y

# verify no leakage
_tr, _te = group_holdout(df)
assert set(_tr['star_id']) & set(_te['star_id']) == set(), 'STAR LEAK across split!'
print(f'group split OK  train={len(_tr)} test={len(_te)}  shared stars=0')

## 3. Shared evaluation

Per handoff Â§10: planets are rare â€” judge on precision/recall + confusion matrix + PR-AUC, **not raw accuracy**. Plus calibration (Brier + reliability curve) because the app reports a confidence number that has to mean something.

In [ ]:
def eval_multiclass(name, model, X_te, y_te, labels):
    """Returns a metrics dict and draws confusion matrix. y_te encoded as label strings."""
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)
    # sklearn orders proba cols by model.classes_ (alphabetical), not by  -> realign or every prob metric is wrong
    classes = list(getattr(model, 'classes_', labels))
    proba = proba[:, [classes.index(c) for c in labels]]
    print(f'\n=== {name} ===')
    print(classification_report(y_te, pred, labels=labels, zero_division=0))
    macro_f1 = f1_score(y_te, pred, labels=labels, average='macro', zero_division=0)
    mcc = matthews_corrcoef(y_te, pred)
    # one-vs-rest PR-AUC per class (the imbalance-honest metric)
    Y = pd.get_dummies(pd.Categorical(y_te, categories=labels))
    pr_auc = {c: average_precision_score(Y[c], proba[:, i]) for i, c in enumerate(labels)}
    try:
        roc = roc_auc_score(Y, proba, average='macro', multi_class='ovr')
    except Exception:
        roc = np.nan
    ConfusionMatrixDisplay(confusion_matrix(y_te, pred, labels=labels),
                           display_labels=labels).plot(values_format='d', cmap='Blues')
    plt.title(name); plt.show()
    return dict(model=name, macro_f1=round(macro_f1,3), mcc=round(mcc,3),
                macro_roc_auc=round(roc,3), **{f'PRAUC_{c}': round(v,3) for c,v in pr_auc.items()})

def plot_reliability(name, y_true_bin, p_pos):
    """Calibration curve + Brier for a binary/one-vs-rest probability."""
    bs = brier_score_loss(y_true_bin, p_pos)
    frac, mean = calibration_curve(y_true_bin, p_pos, n_bins=10, strategy='quantile')
    plt.plot([0,1],[0,1],'k--',lw=1); plt.plot(mean, frac, 'o-')
    plt.xlabel('predicted prob'); plt.ylabel('observed freq')
    plt.title(f'{name} reliability (Brier={bs:.3f})'); plt.show()
    return bs

## 4. Model A â€” Per-dip classifier (CORE) `transit / EB / blend`

Multiclass, mutually exclusive per dip. `unknown` rows are excluded here (they belong to the OOD gate, Â§6).
Candidate models compared head-to-head with identical leakage-safe sampling; pick the winner on macro-F1 / PR-AUC, not accuracy.

In [ ]:
LABELS = ['transit','EB','blend']
known = df[df['dip_class'].isin(LABELS)].copy()
tr, te = group_holdout(known)
Xtr, ytr = tr[DIP_FEATURES].values, tr['dip_class'].values
Xte, yte = te[DIP_FEATURES].values, te['dip_class'].values
Xtr_rs, ytr_rs = resample_train(Xtr, ytr)
print(f'train {len(ytr)} -> resampled {len(ytr_rs)} | test {len(yte)}')

def candidates():
    m = {'RandomForest': RandomForestClassifier(n_estimators=400, class_weight='balanced',
                                                random_state=RNG, n_jobs=-1),
         'HistGBT': HistGradientBoostingClassifier(random_state=RNG)}
    if HAVE_LGBM:
        m['LightGBM'] = LGBMClassifier(n_estimators=500, class_weight='balanced',
                                       random_state=RNG, verbose=-1)
    if HAVE_XGB:
        m['XGBoost'] = XGBClassifier(n_estimators=500, eval_metric='mlogloss',
                                     random_state=RNG, tree_method='hist')
    return m

results_A = []
fitted = {}
for name, mdl in candidates().items():
    if name == 'XGBoost':  # xgboost needs int labels
        from sklearn.preprocessing import LabelEncoder
        le = LabelEncoder().fit(LABELS)
        mdl.fit(Xtr_rs, le.transform(ytr_rs))
        class _W:  # thin wrapper so eval gets string labels + predict_proba
            def __init__(s, m, le): s.m, s.le, s.classes_ = m, le, le.classes_
            def predict(s, X): return s.le.inverse_transform(s.m.predict(X))
            def predict_proba(s, X): return s.m.predict_proba(X)
        mdl = _W(mdl, le)
    else:
        mdl.fit(Xtr_rs, ytr_rs)
    fitted[name] = mdl
    results_A.append(eval_multiclass(name, mdl, Xte, yte, LABELS))

pd.DataFrame(results_A).set_index('model')

In [ ]:
# Sector-split sanity: does the winner survive held-out sectors (the 'future data' test)?
best_name = pd.DataFrame(results_A).set_index('model')['macro_f1'].idxmax()
print('best by macro-F1:', best_name)
s_tr, s_te = sector_holdout(known, test_sectors=[max(known['sector'])])
base = candidates()[best_name] if best_name != 'XGBoost' else candidates()['HistGBT']
Xs, ys = resample_train(s_tr[DIP_FEATURES].values, s_tr['dip_class'].values)
base.fit(Xs, ys)
_ = eval_multiclass(f'{best_name}-style @ held-out sector', base,
                    s_te[DIP_FEATURES].values, s_te['dip_class'].values, LABELS)

In [ ]:
# Calibration of the winner's transit probability (the number the UI shows).
# Wrap with isotonic calibration fit on a held-out slice via CV (handoff Â§10).
from sklearn.base import clone
if best_name in ('RandomForest','HistGBT','LightGBM'):
    cal = CalibratedClassifierCV(candidates()[best_name], method='isotonic', cv=3)
    cal.fit(Xtr_rs, ytr_rs)
    p_transit = cal.predict_proba(Xte)[:, list(cal.classes_).index('transit')]
    _ = plot_reliability(f'{best_name} (isotonic) transit', (yte=='transit').astype(int), p_transit)
else:
    print('skip calibration demo for XGBoost wrapper; same CalibratedClassifierCV pattern applies')

## 5. Model B â€” Spot / rotation detector (binary)

Different question, different representation (full-curve LS/ACF features, Â§3.2). One verdict **per star**, so dedupe to star level and split by star. Handoff says this can even be a tuned threshold â€” so compare a 1-feature threshold rule against a small classifier and only keep the model if it actually beats the rule.

In [ ]:
stars = df.drop_duplicates('star_id')[['star_id','sector'] + SPOT_FEATURES + ['rotation']].copy()
b_tr, b_te = group_holdout(stars, test_size=0.25)
Xb_tr, yb_tr = b_tr[SPOT_FEATURES].values, b_tr['rotation'].values
Xb_te, yb_te = b_te[SPOT_FEATURES].values, b_te['rotation'].values

# Baseline rule: threshold on ls_power, tuned on train to maximize F1.
ls_tr = b_tr['ls_power'].values
thr = max(np.linspace(ls_tr.min(), ls_tr.max(), 200),
          key=lambda t: f1_score(yb_tr, (ls_tr > t).astype(int), zero_division=0))
rule_pred = (b_te['ls_power'].values > thr).astype(int)
print(f'threshold rule  ls_power>{thr:.2f}  test F1={f1_score(yb_te, rule_pred):.3f}  MCC={matthews_corrcoef(yb_te, rule_pred):.3f}')

# Small classifier
clf_b = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RNG, n_jobs=-1)
clf_b.fit(Xb_tr, yb_tr)
p_b = clf_b.predict_proba(Xb_te)[:, 1]
print(f'RF classifier   test F1={f1_score(yb_te, clf_b.predict(Xb_te)):.3f}  MCC={matthews_corrcoef(yb_te, clf_b.predict(Xb_te)):.3f}  ROC-AUC={roc_auc_score(yb_te, p_b):.3f}')
_ = plot_reliability('spot RF', yb_te, p_b)

## 6. Model C â€” OOD / novelty gate

The generalization claim lives here. **Train on inliers only** (the known train fold from Model A). Evaluate on a test set mixing held-out known dips (should pass) with `unknown` dips (should be flagged). Two scorers: Isolation Forest and Mahalanobis distance. Metric = how well the novelty score separates known vs unknown (ROC-AUC / PR-AUC), since there's no per-sample label at deploy time.

In [ ]:
# Inliers = training known dips. Test = held-out known + all unknown.
ood_train = tr[DIP_FEATURES].values  # known classes, train fold only
unknown = df[df['dip_class'] == 'unknown']
ood_test = np.vstack([te[DIP_FEATURES].values, unknown[DIP_FEATURES].values])
is_novel = np.r_[np.zeros(len(te)), np.ones(len(unknown))].astype(int)  # 1 = should be flagged

scaler = StandardScaler().fit(ood_train)
Ztr, Zte = scaler.transform(ood_train), scaler.transform(ood_test)

iso = IsolationForest(n_estimators=300, contamination='auto', random_state=RNG).fit(Ztr)
iso_score = -iso.score_samples(Zte)  # higher = more anomalous

cov = EmpiricalCovariance().fit(Ztr)
maha = cov.mahalanobis(Zte)          # higher = farther from inlier centroid

for nm, sc in [('IsolationForest', iso_score), ('Mahalanobis', maha)]:
    print(f'{nm:16s} novelty ROC-AUC={roc_auc_score(is_novel, sc):.3f}  PR-AUC={average_precision_score(is_novel, sc):.3f}')

# Pick an abstention threshold at a chosen inlier-retention (e.g. keep 95% of known dips).
known_scores = iso_score[is_novel == 0]
thr_ood = np.quantile(known_scores, 0.95)
flagged = iso_score > thr_ood
print(f'\n@95% inlier retention: threshold={thr_ood:.3f}')
print(f'  unknown caught (recall): {flagged[is_novel==1].mean():.2%}')
print(f'  known wrongly flagged  : {flagged[is_novel==0].mean():.2%}')

plt.hist(iso_score[is_novel==0], bins=30, alpha=.6, label='known', density=True)
plt.hist(iso_score[is_novel==1], bins=30, alpha=.6, label='unknown', density=True)
plt.axvline(thr_ood, color='k', ls='--'); plt.legend(); plt.title('IsolationForest novelty score'); plt.show()

## 7. Injection-recovery â€” completeness vs parameters (Â§10)

The defensible form of "works on any dataset in the envelope": measure transit **recall as a function of depth and SNR**. Here it runs on the feature table (recall of the `transit` class binned by depth/SNR). With the real pipeline, replace with injecting synthetic `batman` transits and measuring recovery end-to-end.

In [ ]:
mdl = fitted[best_name]
te2 = te.copy()
te2['pred'] = mdl.predict(Xte)
tr_only = te2[te2['dip_class'] == 'transit'].copy()
tr_only['recovered'] = (tr_only['pred'] == 'transit').astype(int)

fig, ax = plt.subplots(1, 2, figsize=(11,4))
for a, col, lbl in [(ax[0],'depth_ppm','transit depth (ppm)'), (ax[1],'snr','SNR')]:
    bins = pd.qcut(tr_only[col], q=5, duplicates='drop')
    comp = tr_only.groupby(bins, observed=True)['recovered'].mean()
    comp.plot(kind='bar', ax=a); a.set_ylabel('recall'); a.set_xlabel(lbl); a.set_ylim(0,1)
    a.set_xticklabels([f'{iv.mid:.0f}' for iv in comp.index], rotation=0)
fig.suptitle('Transit completeness vs parameter'); plt.tight_layout(); plt.show()

## 8. Scoreboard

In [ ]:
print('MODEL A â€” per-dip classifier candidates:')
display(pd.DataFrame(results_A).set_index('model'))
print(f'\nWinner: {best_name}')
print('MODEL B â€” spot detector: keep RF only if its F1 beats the threshold rule above.')
print('MODEL C â€” OOD gate: pick scorer by novelty ROC-AUC; set abstention threshold by inlier retention.')
print('\nNext: lock the winner + its hyperparams, then wire into the pipeline core (handoff Â§11).')